# 04. Published pruning methods, without retraining

We test three diferent structured pruning methods under our setup, none of them is retrained: CORP, which has no public code and is a reimplementation from its paper (`deitc/pruning.py`), GRAIL from its authors' code, and the pruning stage of VBP from its authors' code, without the fine-tuning its paper adds. All of them run through `scripts/prune/prune_points.py`, which keeps every model as a real FP16 and FP32 file and scores it on val, V2 and R. VBP has AGPL-3.0 licence and stays outside the public repo.

With `RERUN = False` this reads `outputs/pruned_summary.json` (63 models).

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt

import glob
import os
import subprocess
import sys

REPO = Path.cwd()
THIRD = Path(os.environ.get("THIRD_PARTY", REPO.parent / "third_party"))   # scripts/setup_third_party.sh
PY = sys.executable                                     # this environment runs our scripts
os.environ.setdefault("PY", PY)
os.environ.setdefault("ADALOG_PY", "python")            # the published methods' environment, see the README

def sh(cmd):
    """Run a command from the repository root, show its output as it comes, and stop if it fails."""
    with subprocess.Popen(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT) as p:
        for line in p.stdout:
            print(line, end="")
    if p.returncode:
        raise RuntimeError(f"exit code {p.returncode}: {cmd}")

def latest(pattern):
    """The newest file under ../third_party that matches `pattern`."""
    found = sorted(glob.glob(str(THIRD / pattern)), key=os.path.getmtime)
    if not found:
        raise FileNotFoundError(pattern)
    return found[-1]

def baselines():
    """The dense baselines of notebook 02, as {model: {"size_mb": ..., "accuracy": ...}}."""
    b = json.loads((Path("outputs") / "baselines_summary.json").read_text())
    return {k: {"size_mb": b["size_mb"][k], "accuracy": a} for k, a in b["accuracy"].items()}

def checkpoint_of_run(name):
    """The checkpoint a FIMA-Q or APHQ-ViT run saved after block reconstruction, read from its log."""
    folder = THIRD / ("APHQ-ViT-protocol" if name.startswith("APHQ") else "FIMA-Q-protocol")
    log = (folder / f"run_{name.replace(' ', '_')}.log").read_text(errors="replace")
    saved = [l.split("Saving checkpoint to ")[1].strip() for l in log.splitlines() if "Saving checkpoint to" in l and "optimsize" in l]
    return str(folder / saved[-1])

RERUN = False
OUT = Path("outputs")
SETS = ["ImageNet-1k val", "ImageNet-V2", "ImageNet-R"]
load = lambda f: json.loads((OUT / f).read_text())
top1 = lambda e, s="ImageNet-1k val": e["accuracy"][s]["top1"]
pr = load("pruned_summary.json")
base = baselines()
s_int8, ti_int8 = base["DeiT-S INT8"], base["DeiT-Ti INT8"]

def group(v):
    return f"{v['method']} {v['scope']}" + ("" if v.get("compensated", True) else ", no compensation") + f", {v.get('calib_images')} images"
def table(pred):
    print(f"{'model':34s} {'sparsity':>8s} {'MB fp16':>8s} " + " ".join(f"{s.split()[-1]:>6s}" for s in SETS) + "  calib s")
    for k, v in sorted(((k, v) for k, v in pr.items() if pred(v)), key=lambda kv: (group(kv[1]), kv[1]["sparsity"])):
        print(f"{k:34s} {v['sparsity']:8.0%} {v['size_mb']:8.2f} " + " ".join(f"{top1(v, s):6.2f}" for s in SETS) + f"  {v.get('calib_seconds', 0):7.1f}")
def chart(pred, title, x="sparsity"):
    fig, ax = plt.subplots(figsize=(7, 4.5))
    gs = {}
    for v in pr.values():
        if pred(v): gs.setdefault(group(v), []).append((v[x] * (100 if x == "sparsity" else 1), top1(v)))
    for g, pts in sorted(gs.items()):
        ax.plot(*zip(*sorted(pts)), "o-", ms=4, label=g)
    if x == "size_mb":
        ax.axhline(top1(ti_int8), ls=":", c="grey", label="DeiT-Ti INT8 (limit)")
        ax.plot(s_int8["size_mb"], top1(s_int8), "k*", ms=10, label="DeiT-S INT8")
    ax.set(xlabel="sparsity (%)" if x == "sparsity" else "file size, FP16 (MB)", ylabel="top-1 on val (%)", title=title)
    ax.legend(fontsize=7); ax.grid(alpha=0.3); plt.show()

## Rerunning (optional)

The cell below rebuilds all 63 models with `scripts/prune/prune_points.py`, which takes a few hours. Calibration images are the first lines of `splits/calib_seed0.txt`: 128 for GRAIL as the paper says, and 4 000 for CORP and VBP.

In [ ]:
if RERUN:
    for s in (0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7):
        prune = f'"{PY}" scripts/prune/prune_points.py'
        for scope in ("mlp", "attn", "both"):
            sh(f"{prune} corp --sparsity {s} --scope {scope}")
        sh(f"{prune} vbp --sparsity {s}")
        for m in ("grail-wanda", "grail-mag-l2"):
            sh(f"{prune} {m} --sparsity {s}")                 # 128 images, as its paper says
            sh(f"{prune} {m} --sparsity {s} --calib 4000")
        sh(f"{prune} grail-mag-l2 --sparsity {s} --no-compensation")
    pr = load("pruned_summary.json")

## CORP, our reimplementation

CORP prunes MLP channels and attention Q/K dimensions then corrects the next layer by ridge regression on calibration statistics. The paper shows no ridge value, so we chose 1e-3 for our experiments.

In [ ]:
PAPER_CORP = {0.1: 78.48, 0.2: 76.78, 0.3: 74.53, 0.4: 70.84, 0.5: 58.37}   # DeiT-S, FP 79.72
table(lambda v: v["method"] == "corp")
print()
for s, p in PAPER_CORP.items():
    k = f"corp-both_s{int(s * 100)}"
    if k in pr: print(f"joint {s:.0%}: ours {top1(pr[k]):6.2f}   paper {p:6.2f}   difference {top1(pr[k]) - p:+5.2f}")
chart(lambda v: v["method"] == "corp", "CORP by scope")

## GRAIL, from the authors' code

GRAIL selects channels with Wanda or magnitude and then compensates by a ridge reconstruction. The authors' ViT module map was pointed at DeiT's MLP layers. The paper says 128 calibration images while the code uses 6 400, so it runs at 128 and at 4 000. The runs without compensation isolate what the reconstruction adds. GRAIL's paper has no DeiT numbers to compare with.

In [ ]:
table(lambda v: v["method"].startswith("grail"))
chart(lambda v: v["method"].startswith("grail"), "GRAIL: selection, compensation and calibration size")

## VBP, pruning stage only

VBP selects MLP neurons by activation variance and folds their mean into the bias. Its paper reports DeiT-S at 50 % before fine-tuning (64.44 on val), which is the comparison here. Its 10-epoch fine-tuning is future work: it needs the full ImageNet training set and about 5 hours per point.

In [ ]:
table(lambda v: v["method"] == "vbp")
k = "vbp_s50"
if k in pr: print(f"\n50 %: ours {top1(pr[k]):.2f}, paper before fine-tuning 64.44")

## The three methods together

Same scope (MLP only) across methods, against sparsity and against real file size. DeiT-S INT8 and DeiT-Ti INT8 show what an unpruned model gives at the same size or smaller.

In [ ]:
mlp = lambda v: v["scope"] == "mlp" and v.get("compensated", True)
chart(mlp, "MLP pruning by method")
chart(lambda v: v["scope"] in ("mlp", "both") and v.get("compensated", True), "Pruning against file size", x="size_mb")
half = {k: top1(v) for k, v in pr.items() if v["sparsity"] == 0.5 and v["scope"] == "mlp"}
for k, a in sorted(half.items(), key=lambda kv: -kv[1]): print(f"{k:34s} {a:6.2f}")

## Under shift

The same models on V2 and R. A method that keeps val but loses more on R is overfitting its calibration statistics to the training distribution.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 4))
for ax, s in zip(axs, SETS[1:]):
    for g in sorted({group(v) for v in pr.values() if mlp(v)}):
        pts = sorted((top1(v), top1(v, s)) for v in pr.values() if mlp(v) and group(v) == g)
        ax.plot(*zip(*pts), "o-", ms=3, label=g)
    ax.set(xlabel="top-1 on val (%)", ylabel=f"top-1 on {s} (%)"); ax.grid(alpha=0.3)
axs[0].legend(fontsize=6); plt.tight_layout(); plt.show()

## Costs

Pruning takes seconds to minutes (CORP's MLP-only time is inflated because `corp_collect` also gathers attention statistics, so it should not be compared with GRAIL's yet). Pruned FP16 models need no special kernel, so their latency is as measured (`scripts/measure/measure_latency.py`, `outputs/latency_pruned_summary.json`), with dense FP16 timed again in the same session. MACs come from `outputs/costs_summary.json`.

In [ ]:
lat = load("latency_pruned_summary.json")
for k, v in lat["results"].items():
    print(f"{k:52s} {v['median_ms']:8.2f} ms  {v['images_per_s']:8.0f} img/s")
costs = load("costs_summary.json")["points"]
print()
for k, v in costs.items():
    if k in pr or k.startswith("DeiT-S FP16"):
        print(f"{k:34s} " + "  ".join(f"{f}={x:.4g}" for f, x in v.items() if isinstance(x, float)))

## Which blocks each method can prune

Pruning sensitivity from `scripts/allocate/sensitivity.py`: one block's MLP (and, for CORP, its Q and K) pruned at a time, the rest left whole, top-1 change on every tenth val image. Block 0's attention is the outlier.

In [ ]:
sens = load("sensitivity_summary.json")["points"]
import numpy as np
for m in ["CORP", "GRAIL Wanda", "GRAIL magnitude", "VBP"]:
    levels = list(sens[m])
    units = [u for u in sens[m][levels[0]] if isinstance(sens[m][levels[0]][u], dict)]
    grid = np.array([[sens[m][l][u]["val"]["top1_change"] for l in levels] for u in units])
    fig, ax = plt.subplots(figsize=(4, 0.22 * len(units) + 1))
    im = ax.imshow(grid, cmap="RdBu", vmin=-abs(grid).max(), vmax=abs(grid).max(), aspect="auto")
    ax.set_xticks(range(len(levels)), levels); ax.set_yticks(range(len(units)), units, fontsize=6)
    ax.set_title(f"{m}: top-1 change on val"); plt.colorbar(im, ax=ax); plt.tight_layout(); plt.show()